# 05 — Trigger Probability & Photon Detection Efficiency  


---
## 1. Imports

In [ ]:
import sys, os, warnings
import numpy as np
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore', category=UserWarning, module='matplotlib')

# Add project root to path so we can import src.*
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.utils.ingestion import DataIngestionConfig, DataIngestionService
from src.avalanche.trigger import TriggerSolver, MultiplicationSolver

print('Imports OK')

---
## 2. Build device and simulator

In [ ]:
config = DataIngestionConfig.from_defaults()
svc = DataIngestionService(config)
sim = svc.build_simulator()
device = sim.device

Vbr, _ = sim.find_breakdown(V_start=20, V_max=80, V_step=1.0)
print(f"Vbr = {Vbr:.1f} V")

---
## 3. The trigger probability equations

The probability that a single carrier initiates a self-sustaining avalanche is governed by the **coupled McIntyre ODEs**:

$$
\frac{dP_e}{dx} = -\alpha(x) \cdot (1 - P_e) \cdot [P_e + P_h - P_eP_h], \quad P_e(W) = 0 \\
\frac{dP_h}{dx} = +\beta(x) \cdot (1 - P_h) \cdot [P_e + P_h - P_eP_h], \quad P_h(0) = 0
$$

The key term is $P_{\text{tr}} = P_e + P_h - P_eP_h$ — the probability that **at least one** carrier triggers. This creates a nonlinear feedback loop:
- Below breakdown: $P_e = P_h = 0$ is the only solution (damped).
- Above breakdown: a nontrivial branch emerges (pitchfork bifurcation).

### 3.1 Non-local (dead-space) version

The trigger solver also handles the non-local case where dead space is included. Instead of using the local $\alpha(x)$, it evaluates at $\alpha(x + l_d)$ — the carrier must travel its dead space before it can ionize:

In [ ]:
# Solve trigger probability at several excess biases
Vex_list = [0.5, 1.0, 2.0, 3.0, 5.0]
x = sim.grid.x
x_um = x * 1e4

plt.figure(figsize=(10, 4))

for Vex in Vex_list:
    Pe, Ph, E = sim.solve_trigger(Vbr + Vex)
    Ptr = Pe + Ph - Pe * Ph  # average trigger probability
    label = f'Vex = {Vex} V'
    plt.plot(x_um, Ptr, lw=2, label=label)
    print(f"Vex = {Vex:+4.1f} V  |  max(Ptr) = {np.max(Ptr):.4f}  |  "
          f"max|E| = {np.max(np.abs(E)):.2e} V/cm")

plt.xlabel('Depth (µm)')
plt.ylabel('Trigger Probability $P_{\\text{tr}}(x)$')
plt.legend(fontsize=9)
plt.grid(alpha=0.3)
plt.title('Average trigger probability vs position')
plt.show()

---
## 4. Multiplication factor via coupled ODEs

The **mean multiplication factor** $M$ uses a separate linear BVP:

$$
\frac{dM_n}{dx} = -\alpha(x)\,(M_n + M_p), \quad M_n(W) = 1 \\
\frac{dM_p}{dx} = +\beta(x)\,(M_n + M_p), \quad M_p(0) = 1
$$

$M = M_n(0)$ is the net gain for electron injection from the left. This is the **first-moment** quantity (mean gain), distinct from the trigger probability (probability of any gain).

In [ ]:
# Compute multiplication factor at several biases
multi_solver = MultiplicationSolver(sim.grid)

V_range = np.arange(Vbr - 10, Vbr + 10, 2.0)
M_vals = []

for V in V_range:
    phi, E, _ = sim.solve_poisson(float(V))
    E_abs = np.abs(E)
    alpha = sim.ionization.alpha_n(E_abs)
    beta = sim.ionization.alpha_p(E_abs)
    M = multi_solver.compute(E_abs, alpha, beta, sim.grid.x)
    M_vals.append(M)

plt.figure(figsize=(8, 4))
plt.semilogy(V_range, M_vals, 'o-', lw=2)
plt.axvline(Vbr, color='r', ls='--', alpha=0.5, label=f'Vbr = {Vbr} V')
plt.xlabel('Bias (V)')
plt.ylabel('Multiplication M')
plt.grid(alpha=0.3)
plt.legend()
plt.title('Multiplication factor from coupled ODE BVP')
plt.show()

---
## 5. Photon Detection Probability

PDP combines three factors:

$$
\text{PDP}(\lambda) = (1 - R) \cdot T_{\text{dead}}(\lambda) \cdot \int_{\text{absorber}} \alpha(\lambda)\, e^{-\alpha(x - x_0)} \cdot P_{\text{trigger}}(x)\, dx
$$

1. **$(1-R)$**: Transmittance through the top surface (Fresnel reflection)
2. **$T_{\text{dead}}(\lambda)$**: Transmission through the dead layer (p+ cap, contact)
3. **Absorption profile**: $\alpha e^{-\alpha(x-x_0)}$ in the InGaAs absorber
4. **$P_{\text{trigger}}(x)$**: Weighted by the depth where the photon is absorbed

In [ ]:
# Compute PDP at different excess biases for 1550 nm
from src.avalanche.pde import PDEModel

pde_model = sim.pde_model

Vex_range = np.arange(0.5, 8.0, 0.5)
PDP_vals = []

for Vex in Vex_range:
    V = Vbr + Vex
    try:
        _, E, Pe, Ph, xl, xr = sim.get_fields(float(V))
        Ptr = Pe + Ph - Pe * Ph
        # Use the PDE model to compute detection efficiency
        pdp = pde_model.compute(sim.grid.x, E, Ptr,
                                 wavelength=1550e-9,
                                 xr=xr, layers=device.layers,
                                 materials=sim.materials)
        PDP_vals.append(pdp)
    except Exception as e:
        print(f"  Vex = {Vex:.1f} V failed: {e}")
        PDP_vals.append(0.0)

plt.figure(figsize=(8, 4))
plt.plot(Vex_range, PDP_vals, 'o-', lw=2, color='green')
plt.xlabel('Excess Bias $V_{ex}$ (V)')
plt.ylabel('PDP at 1550 nm')
plt.grid(alpha=0.3)
plt.title('Photon Detection Probability vs excess bias')
plt.show()

### PDP spectrum across wavelengths

In [ ]:
# PDP spectrum at fixed excess bias
wavelengths_nm = np.arange(900, 1700, 25)
PDP_spec = []
V_ex = 3.0

Pe, Ph, E = sim.solve_trigger(Vbr + V_ex)
Ptr = Pe + Ph - Pe * Ph

for lam_nm in wavelengths_nm:
    try:
        pdp = pde_model.compute(sim.grid.x, E, Ptr,
                                 wavelength=lam_nm * 1e-9,
                                 xr=0.0, layers=device.layers,
                                 materials=sim.materials)
        PDP_spec.append(pdp)
    except Exception:
        PDP_spec.append(0.0)

plt.figure(figsize=(8, 4))
plt.plot(wavelengths_nm, PDP_spec, 'o-', lw=2, color='purple')
plt.axvline(1550, color='r', ls='--', alpha=0.4, label='1550 nm (telecom)')
plt.axvline(1310, color='orange', ls='--', alpha=0.4, label='1310 nm')
plt.xlabel('Wavelength (nm)')
plt.ylabel('PDP')
plt.grid(alpha=0.3)
plt.legend()
plt.title(f'PDP spectrum at Vex = {V_ex} V')
plt.show()

---## 7. Interactive explorationDrag excess bias to see trigger probability and multiplication change.

In [ ]:
from ipywidgets import FloatSlider, interactive, VBox, Outputfrom IPython.display import displayVex_slider = FloatSlider(value=2.0, min=0.1, max=8.0, step=0.1,                         description='Vex (V):', continuous_update=False,                         style={'description_width': '80px'},                         layout={'width': '400px'})out = Output()@out.capture(clear_output=True)def plot_trigger(Vex):    Vb = Vbr + Vex    phi, E, Pe, Ph, trigger_prob, xr = sim.get_fields(float(Vb))        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3.5))        x = sim.grid.x * 1e4    ax1.plot(x, Pe, 'b-', lw=2, label=r'$P_e(x)$')    ax1.plot(x, Ph, 'r-', lw=2, label=r'$P_h(x)$')    ax1.axvline(xr * 1e4, color='gray', ls='--', alpha=0.5, label='reference')    ax1.set_xlabel('Depth (µm)')    ax1.set_ylabel('Trigger Probability')    ax1.grid(alpha=0.3)    ax1.legend()    ax1.set_title(f'Trigger Probability at Vex = {Vex:.1f} V')        V_range = np.arange(Vbr + 0.5, Vbr + Vex + 1, 0.5) if Vex < 1 else np.arange(Vbr + 0.5, Vbr + 10, 0.5)    P_list = []    for V in V_range:        _, _, _, _, Pt, _ = sim.get_fields(float(V))        P_list.append(Pt)        ax2.plot(V_range - Vbr, P_list, 'g-', lw=2)    ax2.axvline(Vex, color='gray', ls='--', alpha=0.5)    ax2.set_xlabel('Excess Bias (V)')    ax2.set_ylabel('P_trigger')    ax2.grid(alpha=0.3)    ax2.set_title(f'P_trigger(Vex); current Vex = {Vex:.1f} V')        plt.tight_layout()    plt.show()        print(f"Vb = {Vb:.1f} V, Vex = {Vex:.1f} V")    print(f"Trigger probability: P_trigger = {trigger_prob:.4f}")widget = interactive(plot_trigger, Vex=Vex_slider)display(VBox([widget.children[0], out]))print("Drag slider to explore trigger probability vs excess bias.")

---
## 6. Summary

- **Trigger probability** solves coupled McIntyre ODEs (nonlinear, pitchfork bifurcation at breakdown).
- **Multiplication factor** solves a linear BVP — the mean gain diverges at breakdown.
- **PDP** combines: surface transmittance × dead-layer transmission × absorption profile × trigger probability.
- InGaAs absorption gives good PDP at 1310–1550 nm (telecom windows).
- Higher $V_{ex}$ increases PDP as the trigger probability saturates toward 1.

These five notebooks cover the full physics chain of the SPAD simulator — from electrostatics through breakdown, ionization, dark current, and photon detection.